# 42. Scrapy Crawlers — `scrapy`
**Python Libraries Lab · Phase 3 · Web & Networking**

_A full framework for crawling many pages politely and fast._

✅ **Runs in Colab** as-is.

When you need hundreds or thousands of pages, writing your own loop of requests and BeautifulSoup gets messy. Scrapy is a framework that handles the crawling for you. It sends many requests at once, retries failures, respects robots dot t x t, slows down when asked, and saves results as JSON or CSV. You only write a spider, a class that says where to start and how to read each page.

**In this notebook:** The notebook creates and serves a small ShopEasy site on this machine. You will write a spider that collects every product across all pages, follows the pagination, and exports the results to a JSON file, then analyse it with pandas.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q scrapy
import importlib.metadata as md
for pkg in ['scrapy']:
    print(pkg, md.version(pkg))

## Step 1 · Make and serve the practice site
Three pages of products, served on this machine at port 8078.

In [ ]:
from pathlib import Path
site = Path("site"); site.mkdir(exist_ok=True)
products = [("Steel Bottle", 499, 4.5, "Kitchen"), ("Yoga Mat", 1299, 4.1, "Fitness"), ("Desk Lamp", 899, 3.9, "Home"),
            ("Notebook", 149, 4.8, "Stationery"), ("Headphones", 2499, 4.3, "Electronics"), ("Water Jug", 349, 3.5, "Kitchen"),
            ("Dumbbells", 1999, 4.6, "Fitness"), ("Table Fan", 1799, 4.0, "Home"), ("Gel Pens", 99, 4.2, "Stationery")]
pages = [products[0:3], products[3:6], products[6:9]]
for i, page in enumerate(pages, 1):
    cards = "\n".join(f'''<div class="product" data-category="{c}">
  <h2 class="name">{n}</h2><span class="price">Rs. {p:,}</span>
  <span class="rating" title="{r} out of 5">{r}</span><a href="/item/{n.lower().replace(' ', '-')}">details</a>
</div>''' for n, p, r, c in page)
    nxt = f'<a class="next" href="page{i + 1}.html">Next page</a>' if i < len(pages) else ""
    (site / f"page{i}.html").write_text(f"<html><head><title>ShopEasy page {i}</title></head><body><h1>ShopEasy deals</h1>{cards}{nxt}</body></html>")
print("Wrote", [p.name for p in sorted(site.glob("*.html"))])

import threading, functools
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler
class Quiet(SimpleHTTPRequestHandler):
    def log_message(self, *a): pass
site_server = ThreadingHTTPServer(("127.0.0.1", 8078), functools.partial(Quiet, directory="site"))
threading.Thread(target=site_server.serve_forever, daemon=True).start()
print("Practice website at http://127.0.0.1:8078/page1.html")

## Step 2 · Write a spider
`parse` yields one item per product, then follows the next-page link.

In [ ]:
%%writefile shop_spider.py
import scrapy

class ShopSpider(scrapy.Spider):
    name = "shop"
    start_urls = ["http://127.0.0.1:8078/page1.html"]
    custom_settings = {"DOWNLOAD_DELAY": 0.2, "AUTOTHROTTLE_ENABLED": True, "ROBOTSTXT_OBEY": False, "LOG_LEVEL": "INFO"}

    def parse(self, response):
        for card in response.css("div.product"):
            yield {
                "name": card.css(".name::text").get(),
                "price": int(card.css(".price::text").re_first(r"[\d,]+").replace(",", "")),
                "rating": float(card.css(".rating::text").get()),
                "category": card.attrib["data-category"],
                "page": response.url.rsplit("/", 1)[-1],
            }
        next_page = response.css("a.next::attr(href)").get()
        if next_page:
            yield response.follow(next_page, callback=self.parse)

## Step 3 · Run the spider
`-O` overwrites the output file. Scrapy prints a summary of what it did.

In [ ]:
!scrapy runspider shop_spider.py -O products.json 2>&1 | grep -E "Scraped from|item_scraped_count|finish_reason|response_status_count/200"

## Step 4 · Analyse the results
The export is ordinary JSON, ready for pandas.

In [ ]:
import pandas as pd
df = pd.read_json("products.json")
print(len(df), "products from", df["page"].nunique(), "pages")
df.groupby("category").agg(products=("name", "count"), avg_price=("price", "mean"), best_rating=("rating", "max")).round(1)

## Step 5 · Try a selector in the Scrapy shell style
`Selector` lets you test CSS and XPath rules on HTML before putting them in a spider.

In [ ]:
from scrapy.selector import Selector
sel = Selector(text=(site / "page3.html").read_text())
print(sel.css("div.product h2::text").getall())
print(sel.xpath('//div[@data-category="Fitness"]/span[@class="price"]/text()').getall())

## ✍️ Your turn
Using `df`, store the **number of products rated 4.5 or higher** in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == 3, f"Expected 3, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = int((df["rating"] >= 4.5).sum())

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Spider**
- **start_urls**
- **parse callback**
- **response.follow**
- **Item / feed export**
- **Throttling**

**Next:** 43. Browser Automation (`playwright · selenium`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/43_selenium_playwright.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install scrapy
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.